# SBE26 Data Processing

Publish path used:

proc_1 -> FV00 and proc_2 -> FV01 into imos_delivery

### Setup

Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

Import local tools

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_SBE26 = imos_converter_module.IMOSNetCDFConverter_SBE26

from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 259    # update per deployment

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    print_details=True,
)

In [ ]:
converter = IMOSNetCDFConverter_SBE26(input_folder="", input_file="", output_dir="")

### IMOS delivery

Locate proc_1 and proc_2 files

In [ ]:
def resolve_stage_dir(path_value):
    stage_dir = Path(str(path_value)).expanduser()
    if not stage_dir.is_absolute():
        stage_dir = (Path.cwd() / stage_dir).resolve()
    return stage_dir

def resolve_stage_file(path_col, file_col, fallback_pattern):
    stage_dir = resolve_stage_dir(_row[path_col])
    cfg_file = _row.get(file_col, None)
    if pd.notna(cfg_file) and str(cfg_file).strip():
        p = stage_dir / str(cfg_file).strip()
        if p.exists():
            return p

    candidates = sorted(stage_dir.glob(fallback_pattern))
    if not candidates:
        candidates = sorted(stage_dir.glob("*.nc"))
    if not candidates:
        raise FileNotFoundError(f"No NetCDF files found in {stage_dir}")
    return candidates[-1]

proc_1_path = resolve_stage_file(
    path_col="proc_1_path",
    file_col="proc_1_file",
    fallback_pattern=f"{_row['location']}_*_SBE26_{int(_row['inst_id'])}_*.nc",
)
proc_2_path = resolve_stage_file(
    path_col="proc_2_path",
    file_col="proc_2_file",
    fallback_pattern=f"{_row['location']}_*_SBE26_{int(_row['inst_id'])}_*.nc",
)

print(f"Using proc_1 file: {proc_1_path}")
print(f"Using proc_2 file: {proc_2_path}")

In [ ]:
def build_delivery_kwargs(input_path, version):
    return {
        "input_nc_path": str(input_path),
        "longitude": float(_row["longitude"]),
        "latitude": float(_row["latitude"]),
        "depth": float(_row.get("nominal_depth", 0.0)),
        "inst_channels": str(_row.get("mooring_channels", "PT")),
        "start_of_good_data": _row.get("deploy_date", None),
        "site_code": str(_row["location"]),
        "version": version,
        "instrument": str(_row["inst_type"]),
        "inst_id": str(int(_row["inst_id"])),
        "location": str(_row["location"]),
        "output_name_mode": "imos",
        "output_stage": "imos_delivery",
        "metadata_row": _row,
        "process_mode": "publish",
    }

proc_1_delivery_kwargs = build_delivery_kwargs(proc_1_path, "00")
proc_2_delivery_kwargs = build_delivery_kwargs(proc_2_path, "01")

print("Ready to publish proc_1 as FV00 and proc_2 as FV01")

In [ ]:
imos_fv00_out = converter.process(**proc_1_delivery_kwargs)
print(f"IMOS FV00 output: {imos_fv00_out}")

imos_fv01_out = converter.process(**proc_2_delivery_kwargs)
print(f"IMOS FV01 output: {imos_fv01_out}")

Write IMOS delivery filenames to metadata table

In [ ]:
imos_file_names = [Path(imos_fv00_out).name, Path(imos_fv01_out).name]
imos_file_value = ";".join(imos_file_names)
_row = update_metadata_file_fields(
    inst_deploy_id,
    {"imos_deliverables_file": imos_file_value},
    output_paths={"imos_deliverables_file": imos_fv01_out},
    working_dir=Path.cwd(),
)
print(f"Updated imos_deliverables_file: {_row['imos_deliverables_file']}")